# Crop Recommendation System - Model Training

This notebook trains and evaluates various machine learning models for crop recommendation based on soil, climate, and market parameters.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import project modules
import sys
sys.path.append('../src')
from data_loader import DataLoader
from data_preprocessing import DataPreprocessor
from models import CropRecommendationModels, ModelComparator
from evaluation import ModelEvaluator, comprehensive_model_evaluation
from visualization import CropVisualization, InteractiveVisualization
from utils import Config, benchmark_models

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load and Preprocess Data

In [ ]:
# Initialize data loader and preprocessor
loader = DataLoader()
preprocessor = DataPreprocessor()

# Load data (replace with your actual file path)
try:
    # Try to load your dataset
    df = loader.load_and_process_data('../data/sample_data.csv')
    print("Data loaded successfully!")
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please export your Numbers file as CSV and place it in the data/ folder")
    df = None

if df is not None:
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    # Display basic info
    display(df.head())
    display(df.describe())
    
    if 'crop' in df.columns:
        print(f"Crop distribution:")
        display(df['crop'].value_counts())

## 2. Data Preprocessing Pipeline

In [ ]:
if df is not None:
    # Run complete preprocessing pipeline
    result = preprocessor.preprocess_pipeline('../data/sample_data.csv')
    
    if result:
        print("Preprocessing completed successfully!")
        print(f"Training set shape: {result['X_train'].shape}")
        print(f"Testing set shape: {result['X_test'].shape}")
        print(f"Feature columns: {result['feature_columns']}")
        print(f"Target classes: {result['target_classes']}")
        
        # Extract processed data
        X_train = result['X_train']
        X_test = result['X_test']
        y_train = result['y_train']
        y_test = result['y_test']
        label_encoder = result['label_encoder']
        scaler = result['scaler']
        processed_data = result['processed_data']
    else:
        print("Preprocessing failed!")
else:
    print("No data available for preprocessing")

## 3. Initialize and Train Models

In [ ]:
if 'result' in locals() and result:
    # Initialize models
    models = CropRecommendationModels()
    models.initialize_models()
    
    print(f"Initialized {len(models.models)} models:")
    for model_name in models.models.keys():
        print(f"- {model_name}")
    
    # Train all models
    print("\nTraining all models...")
    trained_models = models.train_all_models(X_train, y_train)
    
    print(f"\nTraining completed!")
    print(f"Best model: {type(models.best_model).__name__}")
    
    # Display model scores
    if models.model_scores:
        scores_df = pd.DataFrame(models.model_scores).T
        scores_df.columns = ['CV Mean Accuracy', 'CV Std']
        scores_df = scores_df.sort_values('CV Mean Accuracy', ascending=False)
        
        print("\nModel Performance (Cross-Validation):")
        display(scores_df.round(4))
else:
    print("No preprocessed data available for model training")

## 4. Model Evaluation

In [ ]:
if 'trained_models' in locals() and trained_models:
    # Initialize evaluator
    evaluator = ModelEvaluator()
    
    # Comprehensive evaluation
    evaluation_results = evaluator.create_evaluation_report(
        trained_models, X_test, y_test, X_train, y_train, label_encoder
    )
    
    print("\n=== EVALUATION SUMMARY ===")
    print(f"Best Model: {evaluation_results['best_model']}")
    print(f"Best Accuracy: {evaluation_results['best_accuracy']:.4f}")
    
    # Display comparison table
    comparison_df = pd.DataFrame(evaluation_results['comparison_table'])
    print("\nModel Comparison:")
    display(comparison_df.round(4))
else:
    print("No trained models available for evaluation")

## 5. Feature Importance Analysis

In [ ]:
if 'models' in locals() and models.best_model:
    # Get feature importance from best model
    feature_importance = models.get_feature_importance(
        models.best_model, 
        result['feature_columns']
    )
    
    if feature_importance is not None:
        print("Feature Importance (Top 10):")
        display(feature_importance.head(10))
        
        # Plot feature importance
        plt.figure(figsize=(12, 8))
        sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
        plt.title('Top 15 Feature Importance', fontsize=16, fontweight='bold')
        plt.xlabel('Importance Score')
        plt.ylabel('Features')
        plt.tight_layout()
        plt.show()
    else:
        print("Feature importance not available for this model")
else:
    print("No model available for feature importance analysis")

## 6. Hyperparameter Tuning (Optional)

In [ ]:
# Hyperparameter tuning for the best model (optional)
if 'models' in locals() and models.best_model:
    # Get the name of the best model
    best_model_name = type(models.best_model).__name__
    
    # Map to our model names
    model_name_map = {
        'RandomForestClassifier': 'Random Forest',
        'XGBClassifier': 'XGBoost',
        'LGBMClassifier': 'LightGBM',
        'GradientBoostingClassifier': 'Gradient Boosting',
        'SVC': 'SVM'
    }
    
    model_key = model_name_map.get(best_model_name, None)
    
    if model_key and model_key in ['Random Forest', 'XGBoost', 'LightGBM', 'SVM']:
        print(f"Performing hyperparameter tuning for {model_key}...")
        
        try:
            tuned_model = models.hyperparameter_tuning(model_key, X_train, y_train)
            
            # Evaluate tuned model
            tuned_results = evaluator.evaluate_classification(
                tuned_model, X_test, y_test, f"Tuned {model_key}", label_encoder
            )
            
            print(f"Tuned model accuracy: {tuned_results['accuracy']:.4f}")
            
            # Compare with original
            original_results = evaluation_results['comparison_results'][model_key]
            print(f"Original model accuracy: {original_results['accuracy']:.4f}")
            print(f"Improvement: {tuned_results['accuracy'] - original_results['accuracy']:.4f}")
            
        except Exception as e:
            print(f"Hyperparameter tuning failed: {e}")
    else:
        print(f"Hyperparameter tuning not implemented for {best_model_name}")
else:
    print("No model available for hyperparameter tuning")

## 7. Ensemble Model

In [ ]:
if 'models' in locals() and models.model_scores:
    # Create ensemble model
    ensemble_model = models.create_ensemble_model(X_train, y_train, top_n=3)
    
    if ensemble_model is not None:
        # Evaluate ensemble model
        ensemble_results = evaluator.evaluate_classification(
            ensemble_model, X_test, y_test, "Ensemble Model", label_encoder
        )
        
        print(f"Ensemble Model Accuracy: {ensemble_results['accuracy']:.4f}")
        
        # Compare with best individual model
        best_accuracy = evaluation_results['best_accuracy']
        improvement = ensemble_results['accuracy'] - best_accuracy
        
        print(f"Best Individual Model Accuracy: {best_accuracy:.4f}")
        print(f"Ensemble Improvement: {improvement:.4f}")
        
        if improvement > 0:
            print("✅ Ensemble model performs better!")
        else:
            print("⚠️ Individual model performs better")
else:
    print("No models available for ensemble creation")

## 8. Save Best Model

In [ ]:
if 'models' in locals() and models.best_model:
    # Save the best model
    model_save_path = '../models/trained_models/best_crop_model.pkl'
    
    try:
        models.save_model(models.best_model, model_save_path)
        print(f"Best model saved to: {model_save_path}")
        
        # Save preprocessing objects
        import joblib
        
        joblib.dump(label_encoder, '../models/trained_models/label_encoder.pkl')
        joblib.dump(scaler, '../models/trained_models/scaler.pkl')
        
        print("Preprocessing objects saved!")
        
        # Save feature columns
        import json
        with open('../models/trained_models/feature_columns.json', 'w') as f:
            json.dump(result['feature_columns'], f)
        
        print("Feature columns saved!")
        
    except Exception as e:
        print(f"Error saving model: {e}")
else:
    print("No model to save")

## 9. Test Predictions

In [ ]:
if 'models' in locals() and models.best_model:
    # Test with sample data
    from utils import create_sample_input, format_prediction_output
    
    # Create sample input
    sample_input = create_sample_input()
    
    print("Sample Input:")
    display(sample_input.T)
    
    # Make prediction
    try:
        # Transform input
        X_new = preprocessor.transform_new_data(sample_input)
        
        # Make prediction
        prediction = models.predict_crop(models.best_model, X_new, label_encoder)
        
        if prediction:
            # Format output
            formatted_output = format_prediction_output(prediction, label_encoder)
            
            print("\n=== PREDICTION RESULT ===")
            print(f"Recommended Crop: {formatted_output['recommended_crop']}")
            print(f"Confidence Score: {formatted_output['confidence_score']:.2%}")
            print(f"Recommendation Level: {formatted_output['recommendation_level']}")
            
            if 'top_3_recommendations' in formatted_output:
                print("\nTop 3 Recommendations:")
                for i, crop in enumerate(formatted_output['top_3_recommendations'], 1):
                    print(f"{i}. {crop['crop']} ({crop['probability']:.2%})")
        else:
            print("Prediction failed!")
            
    except Exception as e:
        print(f"Error making prediction: {e}")
else:
    print("No model available for prediction")

## 10. Model Performance Summary

In [ ]:
if 'evaluation_results' in locals():
    print("=== MODEL TRAINING SUMMARY ===")
    print(f"Dataset Shape: {df.shape}")
    print(f"Training Set: {X_train.shape}")
    print(f"Testing Set: {X_test.shape}")
    print(f"Number of Features: {len(result['feature_columns'])}")
    print(f"Number of Crop Classes: {len(result['target_classes'])}")
    
    print(f"\nBest Model: {evaluation_results['best_model']}")
    print(f"Best Accuracy: {evaluation_results['best_accuracy']:.4f}")
    
    # Top 5 models
    comparison_df = pd.DataFrame(evaluation_results['comparison_table'])
    top_5 = comparison_df.head(5)
    
    print("\nTop 5 Performing Models:")
    display(top_5.round(4))
    
    print("\n=== RECOMMENDATIONS ===")
    print("1. Use the best model for production deployment")
    print("2. Consider ensemble methods for improved accuracy")
    print("3. Monitor model performance with new data")
    print("4. Retrain models periodically with updated data")
    print("5. Consider feature engineering for better performance")
    
else:
    print("No evaluation results available")

print("\nModel Training Complete! 🚀")